# EXPLORATION - NOT MAIN PORTFOLIO
# DATE: 4/3/26

# FINDINGS TO BE ADDED TO MAIN:
- random forest performed so much better than logistic regression, knn, and decision tree
- SVM is not suitable for large datasets, not going to be used
- reduced V features dataset slightly improved model performance

## Goal

Testing different classification models to see which model gave the highest performance using the recall performance metric. The dataset that is used will be the one after feature engineering and SMOTE oversampling. Final outcome is to decide which model (and the parameter) will be the final model to be used for credit card fraud detection

## Setup

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('creditcard.csv')

data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


as mentioned before, we will use SMOTE oversample before testing every model. We will test the model on two data, one with full features and one with reduced V features. This is because V features are the result of PCA and supposedly hold important information. If some of the low score V features are marked as unimportant across all the model, we will removed them in the final model

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, \
confusion_matrix

from imblearn.over_sampling import SMOTE

In [16]:
#initial data
X = data.drop('Class', axis=1)
y = data['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

In [18]:
#SMOTE oversample
smote_sample = SMOTE(random_state=1)
X_train, y_train = smote_sample.fit_resample(X_train, y_train)

we reuse the same variable name rather than create a new one since we will be using this for the final model and not the initial data

## Model Experiment

For this comparison, we will only use 1 parameter for each model but we will be using a reasonable starting parameter to ensure fair comparison. Moreover, we will pick 2-3 best performed model for the next experiment, which are hyperparameter tuning, to ensure we won't miss out on any potentially better model & parameter for the final model.

The selection criteria will be using evaluation metric:
1. Recall (primary)
2. F1 Score
3. Precision

### Models

In [6]:
models = {
    'Logistic Regression': LogisticRegression(
        C=1.0,
        penalty='l2', #L2 regularization
        solver='lbfgs',
        max_iter=1000,
        random_state=1,
        n_jobs=-1
    ),

    'Decision Tree': DecisionTreeClassifier(
        max_depth=20, #prevent overfitting
        min_samples_split=10,
        criterion='gini',
        random_state=1
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=20, #prevent overfitting
        min_samples_split=5,
        max_features='sqrt',
        bootstrap=True,
        random_state=1,
        n_jobs=-1
    ),

    'KNN': KNeighborsClassifier(
        n_neighbors=5,
        weights='distance',
        metric='minkowski',
        n_jobs=1
    )
}

After several attempt, I just decided to remove SVM completely as it takes too much time to compute and it has severe computational constraints with large datasets, especially since I use SMOTE which increase the data to 400k records. 

### Train & Evaluate

In [7]:
results = []
trained_models = {}
predictions = {}

print('Training & Evaluating Models')

for name, model in models.items():
    print(f'Training: {name}')

    #calculate training time
    start_time = time.time()

    #train on SMOTE data
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    #calculate metrics
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    #confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    #results
    results.append({
        'Model': name,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Training Time (s)': training_time,
        'True Positives': tp,
        'False Positives': fp,
        'False Negatives': fn,
        'True Negatives': tn
    })

    trained_models[name] = model
    predictions[name] = y_pred

    #print report & confusion matrix
    print("Classification Report")
    print(classification_report(y_test, y_pred))
    print('Confusion Matrix')
    print(cm)

Training & Evaluating Models
Training: Logistic Regression
Classification Report
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     56864
           1       0.11      0.91      0.20        98

    accuracy                           0.99     56962
   macro avg       0.56      0.95      0.60     56962
weighted avg       1.00      0.99      0.99     56962

Confusion Matrix
[[56165   699]
 [    9    89]]
Training: Decision Tree
Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.37      0.87      0.52        98

    accuracy                           1.00     56962
   macro avg       0.68      0.93      0.76     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix
[[56717   147]
 [   13    85]]
Training: Random Forest
Classification Report
              precision    recall  f1-score   support

           0       1.0

In [9]:
result = pd.DataFrame(results)
result

,Model,Precision,Recall,F1-Score,Training Time (s),True Positives,False Positives,False Negatives,True Negatives
0,Logistic Regression,0.112944,0.908163,0.200903,39.226069,89,699,9,56165
1,Decision Tree,0.366379,0.867347,0.515152,39.922562,85,147,13,56717
2,Random Forest,0.826923,0.877551,0.851485,72.194864,86,18,12,56846
3,KNN,0.018265,0.530612,0.035314,0.103414,52,2795,46,54069


just based on this result, we can clearly see that Random Forest performed way better than the other model. Key Findings:
- Random Forest performed best with recall score 87.7%, precision 82% and f1 85%.
- LogReg has the best recall score, however much worse f1 score of 20% and precision 11%
- Decision tree performed better than LogReg but the precision and f1 score are still too low to be considered
- KNN performed very poorly

### Reducing V features
as mentioned before, we will see if we can remove some less important V features. If the model can produced the same or improved performance, we will remove them

In [13]:
rf_model = trained_models['Random Forest']
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

importance_df

,Feature,Importance
14,V14,0.233427
4,V4,0.138636
10,V10,0.116270
12,V12,0.090952
17,V17,0.083075
11,V11,0.073749
16,V16,0.051012
3,V3,0.042473
2,V2,0.025509
7,V7,0.021662


In [17]:
#low important from experiments before
low_score_v = ['V22', 'V23', 'V25', 'V26', 'V28', 'V15', 'V24', 'V13', 'V27']

low_score_v

['V22', 'V23', 'V25', 'V26', 'V28', 'V15', 'V24', 'V13', 'V27']

In [15]:
#double check with decision tree
dt_model = trained_models['Decision Tree']
importance_df2 = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

importance_df2

,Feature,Importance
14,V14,0.811605
4,V4,0.051371
12,V12,0.025834
3,V3,0.014450
0,Time,0.012219
1,V1,0.009598
17,V17,0.007751
10,V10,0.007621
11,V11,0.007479
24,V24,0.005209


even though it is not perfectly the same, we can see some similarity on the low important V features. We will try removing this v features and training the model again

In [19]:
#reduced v
reduced_v = data.drop(columns=low_score_v)
X_redv = reduced_v.drop('Class', axis=1)
y_redv = reduced_v['Class']

X_train_v, X_test_v, y_train_v, y_test_v = \
train_test_split(X_redv, y_redv, test_size=0.2, stratify=y, random_state=1)

#SMOTE resample
X_train_v, y_train_v = smote_sample.fit_resample(X_train_v, y_train_v)

In [20]:
rf_model.fit(X_train_v, y_train_v)
y_pred_v = rf_model.predict(X_test_v)

print("Classification Report")
print(classification_report(y_test_v, y_pred_v))
print("Confusion Matrix")
print(confusion_matrix(y_test_v, y_pred_v))

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.84      0.89      0.87        98

    accuracy                           1.00     56962
   macro avg       0.92      0.94      0.93     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix
[[56848    16]
 [   11    87]]


Since there is an improvement of recall from 88% to 89%, even though slightly, we will continue with the reduced V dataset

## Summary

### Conclusion:
- Random Forest performed better than three other classification models
- Decision Tree and LogReg have pretty high recall score but fallout on f1 and precision score
- KNN scored poorly allover
- We decided to remove SVM completely as the computational resource needed is not feasible
- Removing low importance V features improved the model slightly

### Next Step:
- Only use random forest for the next experiment, hyperparameter tuning
- Use reduced V features dataset